# EEG Motor Imagery Classification Experiments

This notebook runs a full experiment pipeline for classifying motor imagery EEG data (Left, Right, Feet, Tongue).

It includes:
- Data loading and preprocessing
- Baseline + imbalanced class experiments
- cWGAN-GP data augmentation
- Results visualization and printing

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from utils.data_utils import load_data
from utils.experiment_utils import run_experiments, run_augmentation_experiments, print_imbalance_results
from utils.generative_utils import train_gan
from utils.plot_utils import plot_results
%matplotlib inline

## 1. Load and Preprocess the Data

Loads the preprocessed EEG epochs and prints basic dataset information.

In [ ]:
X, y = load_data(Path('data'))
print(f'Loaded {len(y)} epochs across {len(np.unique(y))} classes')

## 2. Run Imbalanced Classification Experiments

Trains classifiers on the original balanced dataset and on multiple imbalanced versions (by removing 50% and 100% of samples from each target class).

In [ ]:
results = run_experiments(
    X, y,
    removal_percentages=[0.5, 1.0],
    target_classes=[0, 1, 2, 3],
    save_dir='results'
)

## 3. Train GAN Generator

Trains a Generative Adversarial Network to generate synthetic EEG samples for data augmentation.

In [ ]:
generator = train_gan(X, y, n_epochs=70, verbose=False)

## 4. Run Augmentation Experiments

Uses the trained GAN to augment the removed samples and evaluates classification performance with augmentation.

In [ ]:
aug_results = run_augmentation_experiments(X, y, results, generator=None)

## 6. Plot Results

Visualizes the performance of the experiments. Saved to `plots` dir by default.

In [ ]:
plot_results(results=results, aug_results=aug_results, save_dir='plots')

## 7. Print Detailed Results Table

Prints a nicely formatted table showing overall and per-class accuracy for all experiments (balanced, removed, and augmented).

In [ ]:
print_imbalance_results(results, aug_results, ["Left", "Right", "Feet", "Tongue"])